In [7]:
# ============================================================
# PYRAMID LOCAL PIPELINE + CORRECTED TARGETED DURATION + ELEVATION
# ============================================================
# Purpose:
#   Build the pyramid dataset using ONLY the local/base pyramids.csv file
#   as the row source.
#
# Removes:
#   - Full Wikipedia table scraping
#   - Wikipedia matching / Wikipedia-only rows
#   - duplicate trimming from Wikipedia
#
# Keeps / Adds:
#   - Time Period from Dynasty
#   - clean/final helper columns
#   - targeted Wikidata/Wikipedia duration research layer
#   - strict Wikipedia title matching to avoid wrong-page duration extraction
#   - range-aware duration parsing
#   - best duration = mean(low, high) when low != high
#   - reign-window duration proxy fallback
#   - unfinished/abandoned/never-built duration safeguards
#   - elevation_m from Latitude/Longitude
#
# Outputs:
#   pyramids_Periods.csv
#   pyramids_kaggle_clean.csv
#   pyramid_elevation_cache.csv
# ============================================================

from __future__ import annotations

from pathlib import Path
import math
import re
import time
import unicodedata

import numpy as np
import pandas as pd
import requests
from tqdm import tqdm

# ------------------------------------------------------------
# 0) USER CONFIG
# ------------------------------------------------------------
PROJECT_DIR = Path("/Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data")

INPUT_CSV = PROJECT_DIR / "pyramids.csv"
FINAL_OUTPUT_CSV = PROJECT_DIR / "pyramids_kaggle_clean.csv"
PERIOD_OUTPUT_CSV = PROJECT_DIR / "pyramids_Periods.csv"
ELEVATION_CACHE_CSV = PROJECT_DIR / "pyramid_elevation_cache.csv"

SAVE_PERIOD_COMPATIBILITY_OUTPUT = True
ENABLE_ELEVATION = True
ENABLE_DURATION_WEB_CHECK = True

# True = keep original columns plus clean/final columns.
# False = export only compact final columns.
KEEP_ORIGINAL_COLUMNS = True

session = requests.Session()
session.headers.update({"User-Agent": "pyramid-local-duration-elevation-pipeline/1.4 academic-use"})

# ------------------------------------------------------------
# 1) HELPERS
# ------------------------------------------------------------
def read_csv_safely(path: Path, **kwargs) -> pd.DataFrame:
    for enc in ["utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError:
            pass
    return pd.read_csv(path, encoding="utf-8", encoding_errors="replace", **kwargs)


def normalize_text(x: object) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.replace("&", " and ")
    s = re.sub(r"\([^)]*\)", " ", s)
    s = re.sub(r"['’`]", "", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\b(the|of|el|al)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def parse_numeric(x: object):
    if pd.isna(x):
        return pd.NA

    s = str(x).strip()
    if not s or s in {"–", "-", "—"}:
        return pd.NA
    if "never built" in s.lower():
        return pd.NA

    s = s.replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", s)

    if not m:
        return pd.NA

    try:
        return float(m.group(0))
    except Exception:
        return pd.NA


def dynasty_to_int(dynasty: object) -> int | None:
    if pd.isna(dynasty):
        return None

    s = str(dynasty).strip().lower()
    if not s:
        return None

    m = re.search(r"\b(\d{1,2})\b", s)
    if not m:
        return None

    try:
        return int(m.group(1))
    except Exception:
        return None


def get_period(dynasty: object) -> str | None:
    d = dynasty_to_int(dynasty)

    if d is None:
        return None
    if 1 <= d <= 2:
        return "Early Dynastic Period (c. 3150–2686 BC)"
    if 3 <= d <= 6:
        return "Old Kingdom (c. 2686–2181 BC)"
    if 8 <= d <= 10:
        return "First Intermediate Period (c. 2181–2055 BC)"
    if 11 <= d <= 14:
        return "Middle Kingdom (c. 2055–1650 BC)"
    if 15 <= d <= 17:
        return "Second Intermediate Period (c. 1650–1550 BC)"
    if d == 18:
        return "New Kingdom (c. 1550–1069 BC)"

    return None


def detect_status(text: object) -> str:
    if pd.isna(text):
        return "unknown"

    t = str(text).lower()

    if "never built" in t:
        return "never_built"
    if "possibly unfinished" in t:
        return "possibly_unfinished"
    if "unfinished" in t:
        return "unfinished"
    if "abandoned" in t:
        return "abandoned"
    if "uncertain" in t:
        return "uncertain"
    if "ruin" in t or "collapsed" in t:
        return "ruined"
    if t.strip():
        return "completed"

    return "unknown"


def first_existing_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None


def move_column_after(df: pd.DataFrame, col: str, after_col: str) -> pd.DataFrame:
    if col not in df.columns or after_col not in df.columns:
        return df

    cols = list(df.columns)
    cols.insert(cols.index(after_col) + 1, cols.pop(cols.index(col)))
    return df[cols]


def chunks(xs: list[int], n: int):
    for i in range(0, len(xs), n):
        yield xs[i:i + n]


def mean_if_range(low: float | int, high: float | int) -> float | int:
    low_f = float(low)
    high_f = float(high)

    if low_f == high_f:
        return int(low_f) if low_f.is_integer() else low_f

    best = (low_f + high_f) / 2
    return int(best) if best.is_integer() else best


# ------------------------------------------------------------
# 2) ELEVATION
# ------------------------------------------------------------
def load_elevation_cache() -> dict[tuple[float, float], float]:
    cache: dict[tuple[float, float], float] = {}

    if ELEVATION_CACHE_CSV.exists():
        c = read_csv_safely(ELEVATION_CACHE_CSV)

        for _, r in c.iterrows():
            lat = pd.to_numeric(r.get("lat"), errors="coerce")
            lon = pd.to_numeric(r.get("lon"), errors="coerce")
            elev = pd.to_numeric(r.get("elevation_m"), errors="coerce")

            if pd.notna(lat) and pd.notna(lon) and pd.notna(elev):
                cache[(round(float(lat), 6), round(float(lon), 6))] = float(elev)

    return cache


def save_elevation_cache(cache: dict[tuple[float, float], float]) -> None:
    rows = [
        {"lat": k[0], "lon": k[1], "elevation_m": v}
        for k, v in sorted(cache.items())
    ]
    pd.DataFrame(rows).to_csv(ELEVATION_CACHE_CSV, index=False)


def fetch_open_elevation_batch(points: list[tuple[float, float]]) -> list[float | None]:
    url = "https://api.open-elevation.com/api/v1/lookup"
    payload = {
        "locations": [
            {"latitude": lat, "longitude": lon}
            for lat, lon in points
        ]
    }

    try:
        r = session.post(url, json=payload, timeout=45)
        r.raise_for_status()

        results = r.json().get("results", []) or []
        vals: list[float | None] = []

        for res in results:
            elev = res.get("elevation")
            vals.append(float(elev) if elev is not None else None)

        if len(vals) == len(points):
            return vals

    except Exception:
        pass

    return [None] * len(points)


def add_elevation(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "elevation_m" not in df.columns:
        df["elevation_m"] = np.nan

    lat_col = first_existing_col(df, ["Latitude", "latitude", "lat"])
    lon_col = first_existing_col(df, ["Longitude", "longitude", "lon", "lng"])

    if lat_col is None or lon_col is None:
        print("Elevation skipped: no Latitude/Longitude columns found.")
        return df

    cache = load_elevation_cache()

    lat = pd.to_numeric(df[lat_col], errors="coerce")
    lon = pd.to_numeric(df[lon_col], errors="coerce")

    missing_idx = df.index[
        df["elevation_m"].isna() & lat.notna() & lon.notna()
    ].tolist()

    print("Elevation candidate rows:", len(missing_idx))

    still_missing: list[int] = []

    for idx in missing_idx:
        key = (round(float(lat.loc[idx]), 6), round(float(lon.loc[idx]), 6))

        if key in cache:
            df.at[idx, "elevation_m"] = cache[key]
        else:
            still_missing.append(idx)

    for batch_idx in tqdm(list(chunks(still_missing, 75)), desc="Open-Elevation"):
        points: list[tuple[float, float]] = []
        keys: list[tuple[int, tuple[float, float]]] = []

        for idx in batch_idx:
            lat_f = float(lat.loc[idx])
            lon_f = float(lon.loc[idx])
            key = (round(lat_f, 6), round(lon_f, 6))

            points.append((lat_f, lon_f))
            keys.append((idx, key))

        elevations = fetch_open_elevation_batch(points)

        for (idx, key), elev in zip(keys, elevations):
            if elev is not None and math.isfinite(elev):
                df.at[idx, "elevation_m"] = elev
                cache[key] = elev

        time.sleep(1.0)

    save_elevation_cache(cache)

    return df


# ------------------------------------------------------------
# 3) DURATION LOGIC
# ------------------------------------------------------------
# Duration hierarchy:
#   1) Manual/direct source-checked duration
#   2) Targeted Wikidata/Wikipedia duration check
#   3) Unfinished / abandoned / never-built classification
#   4) Pharaoh reign-window proxy upper bound
#   5) Unknown

MANUAL_DURATION_FIXES = {
    "great pyramid giza": {
        "low": 26,
        "high": 26,
        "status": "manual_source_checked_estimate",
        "method": "manual source-checked construction-duration estimate",
        "confidence": "medium",
        "source_layer": "manual_duration_fix",
        "statement": "Manual source-checked construction duration estimate: about 26 years.",
        "source_urls": "",
        "caution": "Treat as an estimate, not an exact engineering duration.",
    },
    "great pyramid of giza": {
        "low": 26,
        "high": 26,
        "status": "manual_source_checked_estimate",
        "method": "manual source-checked construction-duration estimate",
        "confidence": "medium",
        "source_layer": "manual_duration_fix",
        "statement": "Manual source-checked construction duration estimate: about 26 years.",
        "source_urls": "",
        "caution": "Treat as an estimate, not an exact engineering duration.",
    },
    "step pyramid djoser": {
        "low": 19,
        "high": 19,
        "status": "manual_source_checked_estimate",
        "method": "manual source-checked construction-duration estimate",
        "confidence": "medium",
        "source_layer": "manual_duration_fix",
        "statement": "Source-checked construction duration estimate: approximately 19 years.",
        "source_urls": "",
        "caution": "Treat as an estimate, not an exact engineering duration.",
    },
    "pyramid of djoser": {
        "low": 19,
        "high": 19,
        "status": "manual_source_checked_estimate",
        "method": "manual source-checked construction-duration estimate",
        "confidence": "medium",
        "source_layer": "manual_duration_fix",
        "statement": "Source-checked construction duration estimate: approximately 19 years.",
        "source_urls": "",
        "caution": "Treat as an estimate, not an exact engineering duration.",
    },
    "pyramid ahmose": {
        "low": 10,
        "high": 30,
        "status": "manual_source_checked_range",
        "method": "manual source-checked construction-duration range",
        "confidence": "medium-low",
        "source_layer": "manual_duration_fix",
        "statement": "Source-checked construction duration range: approximately 10–30 years; best estimate is the mean.",
        "source_urls": "",
        "caution": "Range is uncertain; best estimate is computed as the mean of low and high.",
    },
    "pyramid of ahmose": {
        "low": 10,
        "high": 30,
        "status": "manual_source_checked_range",
        "method": "manual source-checked construction-duration range",
        "confidence": "medium-low",
        "source_layer": "manual_duration_fix",
        "statement": "Source-checked construction duration range: approximately 10–30 years; best estimate is the mean.",
        "source_urls": "",
        "caution": "Range is uncertain; best estimate is computed as the mean of low and high.",
    },
    "pyramid of senusret iii": {
        "low": 20,
        "high": 20,
        "status": "manual_source_checked_estimate",
        "method": "manual source-checked construction-duration estimate",
        "confidence": "medium-low",
        "source_layer": "manual_duration_fix",
        "statement": "Source-checked construction duration estimate: approximately 20 years.",
        "source_urls": "",
        "caution": "Treat as an estimate, not an exact engineering duration.",
    },
    "pyramid senusret iii": {
        "low": 20,
        "high": 20,
        "status": "manual_source_checked_estimate",
        "method": "manual source-checked construction-duration estimate",
        "confidence": "medium-low",
        "source_layer": "manual_duration_fix",
        "statement": "Source-checked construction duration estimate: approximately 20 years.",
        "source_urls": "",
        "caution": "Treat as an estimate, not an exact engineering duration.",
    },
    "red pyramid": {
        "low": 10,
        "high": 17,
        "status": "manual_source_checked_range",
        "method": "manual source-checked construction-duration range",
        "confidence": "medium-low",
        "source_layer": "manual_duration_fix",
        "statement": "Source-checked construction duration range: approximately 10–17 years; best estimate is the mean.",
        "source_urls": "",
        "caution": "Range is uncertain; best estimate is computed as the mean of low and high.",
    },
}

# Upper-bound proxy only, not direct construction duration.
PHARAOH_REIGN_LENGTHS = {
    "djoser": 19,
    "sekhemkhet": 6,
    "khaba": 6,
    "sneferu": 24,
    "khufu": 26,
    "djedefre": 8,
    "khafre": 26,
    "menkaure": 18,
    "shepseskaf": 4,
    "userkaf": 8,
    "sahure": 13,
    "neferirkare kakai": 20,
    "neferirkare": 20,
    "nirefere": 3,
    "raneferef": 3,
    "nyuserre ini": 30,
    "nyuserre": 30,
    "menkauhor kaiu": 8,
    "menkauhor": 8,
    "djedkare isesi": 38,
    "djedkare": 38,
    "unas": 30,
    "teti": 12,
    "pepi i": 49,
    "merenre nemtyemsaf i": 9,
    "merenre": 9,
    "pepi ii": 64,
    "mentuhotep ii": 51,
    "amenemhat i": 29,
    "amenemhet i": 29,
    "senusret i": 45,
    "sesostris i": 45,
    "amenemhat ii": 35,
    "amenemhet ii": 35,
    "senusret ii": 19,
    "sesostris ii": 19,
    "senusret iii": 39,
    "sesostris iii": 39,
    "amenemhat iii": 45,
    "amenemhet iii": 45,
    "amenemhat iv": 9,
    "amenemhet iv": 9,
    "ahmose i": 25,
    "ahmose": 25,
    "khendjer": 4,
    "ameny qemau": 2,
}


def duration_blank_row() -> dict:
    return {
        "construction_duration_low_years": np.nan,
        "construction_duration_high_years": np.nan,
        "construction_duration_best_years": np.nan,
        "duration_status": "duration_unknown",
        "duration_method": "no usable duration evidence found",
        "duration_confidence": "unknown",
        "duration_source_layer": "none",
        "duration_statement": np.nan,
        "source_urls": np.nan,
        "duration_caution": "No direct construction duration or defensible proxy was available.",
    }


def make_duration_row(
    low,
    high,
    status: str,
    method: str,
    confidence: str,
    source_layer: str,
    statement: str,
    source_urls=np.nan,
    caution: str = "",
) -> dict:
    low_num = pd.to_numeric(low, errors="coerce")
    high_num = pd.to_numeric(high, errors="coerce")

    if pd.isna(low_num) or pd.isna(high_num):
        best = np.nan
    else:
        best = mean_if_range(low_num, high_num)

    return {
        "construction_duration_low_years": low_num,
        "construction_duration_high_years": high_num,
        "construction_duration_best_years": best,
        "duration_status": status,
        "duration_method": method,
        "duration_confidence": confidence,
        "duration_source_layer": source_layer,
        "duration_statement": statement,
        "source_urls": source_urls,
        "duration_caution": caution,
    }


def is_unfinished_or_abandoned(row: pd.Series) -> bool:
    fields = []

    for col in [
        "Comment",
        "Notes",
        "Status",
        "status_final",
        "Modern name",
        "name_final",
    ]:
        if col in row.index and pd.notna(row.get(col)):
            fields.append(str(row.get(col)))

    text = " ".join(fields).casefold()

    unfinished_terms = [
        "unfinished",
        "possibly unfinished",
        "abandoned",
        "never built",
        "earthwork just started",
        "ruined before completion",
    ]

    return any(term in text for term in unfinished_terms)


def manual_duration_lookup(row: pd.Series) -> dict | None:
    name = normalize_text(row.get("name_final", row.get("Modern name", "")))

    if name in MANUAL_DURATION_FIXES:
        d = MANUAL_DURATION_FIXES[name]
        low = d["low"]
        high = d["high"]
        best = mean_if_range(low, high)

        statement = d["statement"]
        if low != high:
            statement = f"{statement} Computed best estimate: ({low} + {high}) / 2 = {best} years."

        return make_duration_row(
            low=low,
            high=high,
            status=d["status"],
            method=d["method"],
            confidence=d["confidence"],
            source_layer=d["source_layer"],
            statement=statement,
            source_urls=d.get("source_urls", ""),
            caution=d["caution"],
        )

    return None


def split_sentences(text: str) -> list[str]:
    text = re.sub(r"\s+", " ", str(text)).strip()
    return re.split(r"(?<=[.!?;])\s+", text)


def is_reign_only_context(sentence: str) -> bool:
    s = sentence.casefold()

    reign_terms = [
        "reign",
        "ruled",
        "rule lasted",
        "his rule",
        "her rule",
        "king's reign",
        "pharaoh's reign",
    ]

    construction_terms = [
        "construction",
        "built",
        "build",
        "building",
        "constructed",
        "erected",
        "pyramid took",
        "complex took",
    ]

    has_reign = any(t in s for t in reign_terms)
    has_construction = any(t in s for t in construction_terms)

    return has_reign and not has_construction


def extract_duration_from_text(text: object) -> dict | None:
    """
    Correct behavior:
      - Single value: low = high = best
      - Range value: low = lower bound, high = upper bound, best = mean(low, high)
      - Reign-only durations are ignored as direct construction evidence.
    """
    if pd.isna(text):
        return None

    sentences = split_sentences(str(text))
    range_candidates = []
    point_candidates = []

    range_patterns = [
        r"\b(?:construction|building|built|constructed|erected|completed|finished|duration|took|lasted)[^.;:]{0,120}?\b(\d{1,3})\s*(?:-|–|—|to)\s*(\d{1,3})\s+years?\b",
        r"\b(?:construction|building|built|constructed|erected|completed|finished|duration|took|lasted)[^.;:]{0,120}?\bbetween\s+(\d{1,3})\s+and\s+(\d{1,3})\s+years?\b",
        r"\b(\d{1,3})\s*(?:-|–|—|to)\s*(\d{1,3})\s+years?\s+(?:to build|to construct|of construction|for construction)\b",
        r"\bbetween\s+(\d{1,3})\s+and\s+(\d{1,3})\s+years?\s+(?:to build|to construct|of construction|for construction)\b",
    ]

    point_patterns = [
        r"\b(?:construction|building|built|constructed|completed|finished|erected)[^.;:]{0,120}?\b(?:over|in|within|after|for|during|lasted|took)\s+(?:about|around|approximately|roughly|nearly|some|an estimated)?\s*(\d{1,3})\s+years?\b",
        r"\b(?:took|lasted)\s+(?:about|around|approximately|roughly|nearly|some|an estimated)?\s*(\d{1,3})\s+years?\b",
        r"\b(?:approximately|about|around|roughly|estimated|an estimated|estimate of)\s+(\d{1,3})\s+years?\b",
        r"\b(\d{1,3})\s+years?\s+(?:to build|to construct|of construction|for construction)\b",
    ]

    for sentence in sentences:
        if is_reign_only_context(sentence):
            continue

        s = sentence.strip()
        s_low = s.casefold()

        has_construction_context = any(
            k in s_low
            for k in [
                "construction",
                "built",
                "build",
                "building",
                "constructed",
                "erected",
                "to build",
                "to construct",
                "pyramid",
                "complex",
            ]
        )

        if not has_construction_context:
            continue

        for pat in range_patterns:
            for m in re.finditer(pat, s, flags=re.IGNORECASE):
                try:
                    a = int(m.group(1))
                    b = int(m.group(2))
                    low = min(a, b)
                    high = max(a, b)

                    if 1 <= low <= high <= 100:
                        range_candidates.append((low, high, s))
                except Exception:
                    pass

        for pat in point_patterns:
            for m in re.finditer(pat, s, flags=re.IGNORECASE):
                try:
                    val = int(m.group(1))
                    if 1 <= val <= 100:
                        point_candidates.append((val, s))
                except Exception:
                    pass

    if range_candidates:
        low, high, evidence_sentence = sorted(range_candidates, key=lambda x: (x[1] - x[0], x[0]))[0]
        best = mean_if_range(low, high)

        return {
            "low": low,
            "high": high,
            "best": best,
            "status": "wikipedia_text_extracted_range",
            "confidence": "medium-low",
            "statement": (
                f"Wikipedia text produced a construction-duration range of about "
                f"{low}–{high} years. Best estimate uses the mean: ({low} + {high}) / 2 = {best} years. "
                f"Evidence sentence: {evidence_sentence}"
            ),
            "caution": "Range extraction is automated; verify before treating as exact.",
        }

    if point_candidates:
        val, evidence_sentence = sorted(point_candidates, key=lambda x: x[0])[0]

        return {
            "low": val,
            "high": val,
            "best": val,
            "status": "wikipedia_text_extracted_estimate",
            "confidence": "medium-low",
            "statement": (
                f"Wikipedia text produced a direct construction-duration estimate of about {val} years. "
                f"Evidence sentence: {evidence_sentence}"
            ),
            "caution": "Point extraction is automated; verify before treating as exact.",
        }

    return None


GENERIC_TITLE_TOKENS = {
    "pyramid",
    "pyramids",
    "of",
    "the",
    "north",
    "south",
    "east",
    "west",
    "giza",
    "saqqara",
    "dahshur",
    "abydos",
    "abusir",
    "egypt",
    "egyptian",
    "king",
    "queen",
}


def distinctive_tokens(text: object) -> set[str]:
    key = normalize_text(text)
    tokens = set(key.split())
    return {
        t for t in tokens
        if len(t) >= 3 and t not in GENERIC_TITLE_TOKENS
    }


def wikipedia_title_is_acceptable(title: str, expected_name: str, pharaoh: str = "") -> bool:
    title_tokens = distinctive_tokens(title)
    name_tokens = distinctive_tokens(expected_name)
    pharaoh_tokens = distinctive_tokens(pharaoh)

    if not title_tokens:
        return False

    if name_tokens and (title_tokens & name_tokens):
        return True

    if pharaoh_tokens and (title_tokens & pharaoh_tokens):
        return True

    return False


def wikipedia_search_page_title(
    query: str,
    expected_name: str = "",
    expected_pharaoh: str = "",
    retries: int = 3,
) -> tuple[str | None, str | None]:
    """
    Strict page resolver:
      - does not accept pages just because they contain "pyramid"
      - requires distinctive overlap with the expected pyramid name or pharaoh
      - prevents wrong matches like Menkaure/Pepi/Djedkare using Great Pyramid page
    """
    api = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "format": "json",
        "list": "search",
        "srsearch": query,
        "srlimit": 10,
    }

    for attempt in range(retries):
        try:
            r = session.get(api, params=params, timeout=25)

            if r.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue

            r.raise_for_status()
            results = r.json().get("query", {}).get("search", []) or []

            accepted = []

            for item in results:
                title = item.get("title")
                snippet = item.get("snippet", "")

                if not title:
                    continue

                blob = f"{title} {snippet}".casefold()

                if "pyramid" not in blob:
                    continue

                if not wikipedia_title_is_acceptable(
                    title=title,
                    expected_name=expected_name,
                    pharaoh=expected_pharaoh,
                ):
                    continue

                title_tokens = distinctive_tokens(title)
                name_tokens = distinctive_tokens(expected_name)
                pharaoh_tokens = distinctive_tokens(expected_pharaoh)

                score = 0
                score += 5 * len(title_tokens & name_tokens)
                score += 3 * len(title_tokens & pharaoh_tokens)

                if "egypt" in blob or "pharaoh" in blob:
                    score += 1

                accepted.append((score, title))

            if accepted:
                accepted = sorted(accepted, reverse=True)
                title = accepted[0][1]
                page_url = "https://en.wikipedia.org/wiki/" + str(title).replace(" ", "_")
                return title, page_url

            return None, None

        except Exception:
            time.sleep(0.5 * (attempt + 1))

    return None, None


def wikipedia_fetch_plaintext(title: str, retries: int = 3) -> str | None:
    api = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "explaintext": 1,
        "redirects": 1,
        "titles": title,
    }

    for attempt in range(retries):
        try:
            r = session.get(api, params=params, timeout=30)

            if r.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue

            r.raise_for_status()
            pages = r.json().get("query", {}).get("pages", {}) or {}

            for _, page in pages.items():
                extract = page.get("extract")
                if isinstance(extract, str) and extract.strip():
                    return extract

        except Exception:
            time.sleep(0.5 * (attempt + 1))

    return None


def wikidata_search_qid_for_pyramid(row: pd.Series, retries: int = 3) -> tuple[str | None, str | None]:
    name = str(row.get("name_final", row.get("Modern name", "")))
    pharaoh = str(row.get("pharaoh_final", row.get("Pharaoh", "")))
    site = str(row.get("site_final", row.get("Site", "")))

    query = f"{name} {pharaoh} {site} pyramid".strip()

    api = "https://www.wikidata.org/w/api.php"

    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": "en",
        "uselang": "en",
        "type": "item",
        "limit": 10,
        "search": query,
    }

    for attempt in range(retries):
        try:
            r = session.get(api, params=params, timeout=25)

            if r.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue

            r.raise_for_status()
            results = r.json().get("search", []) or []

            name_key = normalize_text(name)

            scored = []
            for item in results:
                qid = item.get("id")
                if not isinstance(qid, str) or not qid.startswith("Q"):
                    continue

                label = item.get("label", "")
                desc = item.get("description", "")
                blob = f"{label} {desc}".casefold()
                label_key = normalize_text(label)

                score = 0
                if "pyramid" in blob:
                    score += 4
                if "egypt" in blob or "egyptian" in blob:
                    score += 1
                if name_key and name_key in label_key:
                    score += 5
                elif name_key and any(part in label_key for part in name_key.split() if len(part) > 3):
                    score += 2

                scored.append((score, qid))

            scored = sorted(scored, reverse=True)

            if scored and scored[0][0] > 0:
                qid = scored[0][1]
                return qid, f"https://www.wikidata.org/wiki/{qid}"

            if results:
                qid = results[0].get("id")
                if isinstance(qid, str) and qid.startswith("Q"):
                    return qid, f"https://www.wikidata.org/wiki/{qid}"

        except Exception:
            time.sleep(0.5 * (attempt + 1))

    return None, None


def parse_wikidata_year(value: object) -> int | None:
    if value is None:
        return None

    s = str(value)
    m = re.match(r"^([+-]?\d{1,6})", s)
    if not m:
        return None

    try:
        return int(m.group(1))
    except Exception:
        return None


def wikidata_try_duration_from_dates(qid: str, retries: int = 3) -> tuple[int | float | None, str | None]:
    """
    Cautious inferred duration from Wikidata date-like fields.

    Checks:
      P571  = inception
      P1619 = date of official opening
    """
    if not qid:
        return None, None

    sparql = f"""
    SELECT ?inception ?opening WHERE {{
      OPTIONAL {{ wd:{qid} wdt:P571 ?inception. }}
      OPTIONAL {{ wd:{qid} wdt:P1619 ?opening. }}
    }}
    LIMIT 5
    """

    url = "https://query.wikidata.org/sparql"

    for attempt in range(retries):
        try:
            r = session.get(
                url,
                params={"query": sparql},
                headers={"Accept": "application/sparql-results+json"},
                timeout=45,
            )

            if r.status_code == 429:
                time.sleep(2.0 * (attempt + 1))
                continue

            r.raise_for_status()
            rows = r.json().get("results", {}).get("bindings", []) or []

            durations = []

            for b in rows:
                inc = b.get("inception", {}).get("value")
                opn = b.get("opening", {}).get("value")

                y1 = parse_wikidata_year(inc)
                y2 = parse_wikidata_year(opn)

                if y1 is None or y2 is None:
                    continue

                dur = abs(y2 - y1)

                if 1 <= dur <= 100:
                    durations.append(dur)

            if durations:
                return min(durations), f"https://www.wikidata.org/wiki/{qid}"

        except Exception:
            time.sleep(0.5 * (attempt + 1))

    return None, None


DURATION_WEB_CACHE: dict[str, dict | None] = {}


def targeted_web_duration_lookup(row: pd.Series) -> dict | None:
    if not ENABLE_DURATION_WEB_CHECK:
        return None

    name = str(row.get("name_final", row.get("Modern name", "")))
    pharaoh = str(row.get("pharaoh_final", row.get("Pharaoh", "")))
    site = str(row.get("site_final", row.get("Site", "")))

    cache_key = normalize_text(f"{name} {pharaoh} {site}")

    if cache_key in DURATION_WEB_CACHE:
        return DURATION_WEB_CACHE[cache_key]

    query = f"{name} {pharaoh} {site} pyramid construction duration".strip()

    # First: Wikidata date-based attempt.
    qid, wd_url = wikidata_search_qid_for_pyramid(row)

    if qid:
        wd_duration, wd_source = wikidata_try_duration_from_dates(qid)

        if wd_duration is not None:
            result = make_duration_row(
                low=wd_duration,
                high=wd_duration,
                status="wikidata_date_estimate",
                method="duration inferred from Wikidata start/end-like date properties",
                confidence="low",
                source_layer="wikidata_duration_check",
                statement=f"Wikidata date fields imply an approximate construction window of {wd_duration} years.",
                source_urls=wd_source,
                caution="This is inferred from Wikidata date fields, not necessarily a direct construction-duration statement.",
            )
            DURATION_WEB_CACHE[cache_key] = result
            return result

    # Second: Wikipedia text phrase/range attempt.
    title, page_url = wikipedia_search_page_title(
        query=query,
        expected_name=name,
        expected_pharaoh=pharaoh,
    )

    if title:
        text = wikipedia_fetch_plaintext(title)
        duration_info = extract_duration_from_text(text)

        if duration_info is not None:
            result = make_duration_row(
                low=duration_info["low"],
                high=duration_info["high"],
                status=duration_info["status"],
                method="duration phrase or range extracted from Wikipedia article text",
                confidence=duration_info["confidence"],
                source_layer="wikipedia_text_duration_check",
                statement=duration_info["statement"],
                source_urls=page_url,
                caution=duration_info["caution"],
            )
            DURATION_WEB_CACHE[cache_key] = result
            return result

    DURATION_WEB_CACHE[cache_key] = None
    return None


def unfinished_duration_row(row: pd.Series) -> dict:
    return {
        "construction_duration_low_years": np.nan,
        "construction_duration_high_years": np.nan,
        "construction_duration_best_years": np.nan,
        "duration_status": "unfinished_duration_unknown",
        "duration_method": "classified from original dataset comments/text",
        "duration_confidence": "medium",
        "duration_source_layer": "dataset_comment",
        "duration_statement": "The pyramid appears unfinished, abandoned, or never built based on the dataset text.",
        "source_urls": np.nan,
        "duration_caution": "Do not treat this as zero years. It means the construction duration is unknown.",
    }


def reign_proxy_duration_row(row: pd.Series) -> dict | None:
    pharaoh = normalize_text(row.get("pharaoh_final", row.get("Pharaoh", "")))

    if pharaoh not in PHARAOH_REIGN_LENGTHS:
        return None

    reign_years = PHARAOH_REIGN_LENGTHS[pharaoh]
    pharaoh_display = row.get("pharaoh_final", row.get("Pharaoh", ""))

    return {
        "construction_duration_low_years": np.nan,
        "construction_duration_high_years": reign_years,
        "construction_duration_best_years": np.nan,
        "duration_status": "reign_window_proxy_only",
        "duration_method": "upper-bound proxy from attributed pharaoh reign length",
        "duration_confidence": "low",
        "duration_source_layer": "reign_chronology_proxy",
        "duration_statement": (
            f"No direct construction duration was found. The pyramid is attributed to "
            f"{pharaoh_display}, so the upper-bound proxy uses that ruler's approximate "
            f"reign length: about {reign_years} years."
        ),
        "source_urls": np.nan,
        "duration_caution": "This is not a direct construction duration. It is only an upper-bound proxy.",
    }


def assign_duration_for_row(row: pd.Series) -> dict:
    # Layer 1: direct/manual source-checked known duration.
    manual = manual_duration_lookup(row)
    if manual is not None:
        return manual

    # Layer 2: targeted Wikidata/Wikipedia duration check.
    web = targeted_web_duration_lookup(row)
    if web is not None:
        return web

    # Layer 3: unfinished / abandoned / never-built.
    if is_unfinished_or_abandoned(row):
        return unfinished_duration_row(row)

    # Layer 4: reign-window proxy upper bound.
    proxy = reign_proxy_duration_row(row)
    if proxy is not None:
        return proxy

    # Layer 5: unknown.
    return duration_blank_row()


def add_duration_fields(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    duration_rows = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Duration research"):
        duration_rows.append(assign_duration_for_row(row))
        time.sleep(0.15)

    duration_df = pd.DataFrame(duration_rows, index=df.index)

    for col in duration_df.columns:
        df[col] = duration_df[col]

    return df


# ------------------------------------------------------------
# 4) BASE PYRAMID DATA
# ------------------------------------------------------------
def load_base_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Could not find input CSV: {path}")

    df = read_csv_safely(path)

    required = {"Modern name", "Pharaoh", "Dynasty", "Site"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(f"Input pyramid CSV missing required columns: {missing}")

    return df


def add_period_and_final_fields(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["Time Period"] = df["Dynasty"].apply(get_period)
    df = move_column_after(df, "Time Period", "Dynasty")

    df["source_base"] = "kaggle"

    df["name_key"] = df["Modern name"].map(normalize_text)
    df["pharaoh_key"] = df["Pharaoh"].map(normalize_text)
    df["site_key"] = df["Site"].map(normalize_text)

    df["name_final"] = df["Modern name"]
    df["pharaoh_final"] = df["Pharaoh"]
    df["dynasty_final"] = df["Dynasty"]
    df["time_period_final"] = df["Time Period"]
    df["site_final"] = df["Site"]

    height_col = first_existing_col(df, ["Height (m)", "Height", "height_m"])
    volume_col = first_existing_col(df, ["Volume (cu.m)", "Volume", "volume_m3"])
    slope_col = first_existing_col(df, ["Slope", "Inclination", "inclination_deg"])
    comment_col = first_existing_col(df, ["Comment", "Notes", "Status"])

    df["height_m_final"] = df[height_col].apply(parse_numeric) if height_col else pd.NA
    df["volume_m3_final"] = df[volume_col].apply(parse_numeric) if volume_col else pd.NA
    df["inclination_deg_final"] = df[slope_col].apply(parse_numeric) if slope_col else pd.NA
    df["status_final"] = df[comment_col].map(detect_status) if comment_col else "unknown"

    return df


def select_final_columns(df: pd.DataFrame) -> pd.DataFrame:
    duration_cols = [
        "construction_duration_low_years",
        "construction_duration_high_years",
        "construction_duration_best_years",
        "duration_status",
        "duration_method",
        "duration_confidence",
        "duration_source_layer",
        "duration_statement",
        "source_urls",
        "duration_caution",
    ]

    if KEEP_ORIGINAL_COLUMNS:
        first_cols = [
            "source_base",
            "name_final",
            "pharaoh_final",
            "dynasty_final",
            "time_period_final",
            "site_final",
            "height_m_final",
            "volume_m3_final",
            "inclination_deg_final",
            "status_final",
            *duration_cols,
            "elevation_m",
            "Modern name",
            "Pharaoh",
            "Dynasty",
            "Time Period",
            "Site",
        ]

        first_cols = [c for c in first_cols if c in df.columns]
        remaining_cols = [c for c in df.columns if c not in first_cols]

        return df[first_cols + remaining_cols]

    compact_cols = [
        "source_base",
        "name_final",
        "pharaoh_final",
        "dynasty_final",
        "time_period_final",
        "site_final",
        "height_m_final",
        "volume_m3_final",
        "inclination_deg_final",
        "status_final",
        *duration_cols,
        "elevation_m",
    ]

    compact_cols = [c for c in compact_cols if c in df.columns]

    return df[compact_cols]


# ------------------------------------------------------------
# 5) RUN PIPELINE
# ------------------------------------------------------------
print("Reading:", INPUT_CSV)

df = load_base_data(INPUT_CSV)
print("Input rows:", len(df), "columns:", df.shape[1])

df = add_period_and_final_fields(df)

df = add_duration_fields(df)

if ENABLE_ELEVATION:
    df = add_elevation(df)

if SAVE_PERIOD_COMPATIBILITY_OUTPUT:
    period_df = df.copy()

    helper_cols = [
        "source_base",
        "name_key",
        "pharaoh_key",
        "site_key",
        "name_final",
        "pharaoh_final",
        "dynasty_final",
        "time_period_final",
        "site_final",
        "height_m_final",
        "volume_m3_final",
        "inclination_deg_final",
        "status_final",
        "construction_duration_low_years",
        "construction_duration_high_years",
        "construction_duration_best_years",
        "duration_status",
        "duration_method",
        "duration_confidence",
        "duration_source_layer",
        "duration_statement",
        "source_urls",
        "duration_caution",
        "elevation_m",
    ]

    period_df = period_df.drop(
        columns=[c for c in helper_cols if c in period_df.columns],
        errors="ignore",
    )

    period_df.to_csv(PERIOD_OUTPUT_CSV, index=False)
    print("Saved period compatibility file:", PERIOD_OUTPUT_CSV)

final = select_final_columns(df)
final.to_csv(FINAL_OUTPUT_CSV, index=False)

print("Saved final pyramid file:", FINAL_OUTPUT_CSV)
print("Final rows:", len(final), "columns:", final.shape[1])

print("\nFinal columns:")
for c in final.columns:
    print(" -", c)

Reading: /Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/pyramids.csv
Input rows: 62 columns: 16


Duration research: 100%|██████████| 62/62 [10:07<00:00,  9.80s/it]


Elevation candidate rows: 62


Open-Elevation: 100%|██████████| 1/1 [00:02<00:00,  2.17s/it]

Saved period compatibility file: /Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/pyramids_Periods.csv
Saved final pyramid file: /Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/pyramids_kaggle_clean.csv
Final rows: 62 columns: 41

Final columns:
 - source_base
 - name_final
 - pharaoh_final
 - dynasty_final
 - time_period_final
 - site_final
 - height_m_final
 - volume_m3_final
 - inclination_deg_final
 - status_final
 - construction_duration_low_years
 - construction_duration_high_years
 - construction_duration_best_years
 - duration_status
 - duration_method
 - duration_confidence
 - duration_source_layer
 - duration_statement
 - source_urls
 - duration_caution
 - elevation_m
 - Modern name
 - Pharaoh
 - Dynasty
 - Time Period
 - Site
 - Ancient name
 - Base1 (m)
 - Base2 (m)
 - Height (m)
 - Slope (dec degr)
 - Volume (cu.m)
 - Latitude
 - Longitude
 - Type
 - Lepsius
 - Material
 - Comment
 - name_key
 - pharaoh_key
 - site_key


In [18]:
from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd


def normalize_text(x: object) -> str:
    if pd.isna(x):
        return ""

    s = str(x).strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.replace("&", " and ")
    s = re.sub(r"\([^)]*\)", " ", s)
    s = re.sub(r"['’`]", "", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\b(the|of|el|al)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s


def mean_if_range(low: float | int, high: float | int) -> float | int:
    low_f = float(low)
    high_f = float(high)

    if low_f == high_f:
        return int(low_f) if low_f.is_integer() else low_f

    best = (low_f + high_f) / 2
    return int(best) if best.is_integer() else best


def read_csv_safely(path: Path, **kwargs) -> pd.DataFrame:
    for enc in ["utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError:
            pass

    return pd.read_csv(path, encoding="utf-8", encoding_errors="replace", **kwargs)


def ensure_duration_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    required_cols = {
        "construction_duration_low_years": np.nan,
        "construction_duration_high_years": np.nan,
        "construction_duration_best_years": np.nan,
        "duration_status": "duration_unknown",
        "duration_method": "no usable duration evidence found",
        "duration_confidence": "unknown",
        "duration_source_layer": "none",
        "duration_statement": np.nan,
        "source_urls": np.nan,
        "duration_caution": np.nan,
    }

    for col, default in required_cols.items():
        if col not in df.columns:
            df[col] = default

    return df


def name_mask(df: pd.DataFrame, names: list[str]) -> pd.Series:
    """
    Matches against name_final first, then Modern name if name_final is unavailable.
    """
    if "name_final" in df.columns:
        source = df["name_final"]
    elif "Modern name" in df.columns:
        source = df["Modern name"]
    else:
        raise ValueError("Could not find either 'name_final' or 'Modern name' column.")

    targets = {normalize_text(x) for x in names}

    return source.map(normalize_text).isin(targets)


def set_manual_direct_duration(
    df: pd.DataFrame,
    names: list[str],
    low: float,
    high: float,
    status: str,
    confidence: str,
    statement: str,
    source_urls: str = "",
) -> pd.DataFrame:
    df = df.copy()

    low_out = int(low) if float(low).is_integer() else float(low)
    high_out = int(high) if float(high).is_integer() else float(high)
    best_out = mean_if_range(low, high)

    mask = name_mask(df, names)

    if not mask.any():
        print("WARNING: no rows matched direct-duration edit:", names)
        return df

    final_statement = statement

    if low_out != high_out:
        final_statement = (
            f"{statement} Best estimate computed as mean: "
            f"({low_out} + {high_out}) / 2 = {best_out} years."
        )

    df.loc[mask, "construction_duration_low_years"] = low_out
    df.loc[mask, "construction_duration_high_years"] = high_out
    df.loc[mask, "construction_duration_best_years"] = best_out
    df.loc[mask, "duration_status"] = status
    df.loc[mask, "duration_method"] = "manual curated construction-duration correction"
    df.loc[mask, "duration_confidence"] = confidence
    df.loc[mask, "duration_source_layer"] = "manual_online_and_book_review"
    df.loc[mask, "duration_statement"] = final_statement
    df.loc[mask, "source_urls"] = source_urls
    df.loc[mask, "duration_caution"] = (
        "Manual edit based on online source review and books/secondary historical material read. "
        "Treat as a curated estimate, not an exact engineering duration."
    )

    return df


def set_manual_reign_proxy(
    df: pd.DataFrame,
    names: list[str],
    proxy_high: float,
    pharaoh_label: str,
    confidence: str = "low",
    caution_extra: str = "",
) -> pd.DataFrame:
    df = df.copy()

    high_out = int(proxy_high) if float(proxy_high).is_integer() else float(proxy_high)

    mask = name_mask(df, names)

    if not mask.any():
        print("WARNING: no rows matched reign-proxy edit:", names)
        return df

    df.loc[mask, "construction_duration_low_years"] = np.nan
    df.loc[mask, "construction_duration_high_years"] = high_out
    df.loc[mask, "construction_duration_best_years"] = np.nan

    df.loc[mask, "duration_status"] = "reign_window_proxy_only"
    df.loc[mask, "duration_method"] = "manual curated upper-bound proxy from attributed pharaoh reign length"
    df.loc[mask, "duration_confidence"] = confidence
    df.loc[mask, "duration_source_layer"] = "manual_online_and_book_review"

    df.loc[mask, "duration_statement"] = (
        f"No defensible direct construction duration was confirmed. "
        f"The pyramid is attributed to {pharaoh_label}, so this row keeps only an upper-bound "
        f"reign-window proxy of about {high_out} years."
    )

    df.loc[mask, "source_urls"] = ""

    df.loc[mask, "duration_caution"] = (
        "Manual edit based on online source review and books/secondary historical material read. "
        "This is not a direct construction duration. It is only a reign-window upper-bound proxy. "
        + caution_extra
    ).strip()

    return df


# ============================================================
# NEW AUTO-FILL REIGN WINDOW PROXY HELPERS
# ============================================================

def first_existing_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None


def extract_reign_proxy_from_text(x: object) -> float:
    """
    Extracts a reign-window length from text if the dataset has a reign/date field.

    Examples:
        "2589–2566 BC" -> 23
        "c. 2494–2487 BC" -> 7
        "19 years" -> 19
        "reigned 26 years" -> 26
    """
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()
    s = s.replace("–", "-").replace("—", "-").replace("−", "-")
    s = re.sub(r"\bc\.?\b|circa|ca\.?", " ", s)
    s = re.sub(r"\s+", " ", s)

    # Direct duration expression: "26 years", "26-year reign"
    m = re.search(r"(\d+(?:\.\d+)?)\s*(?:years?|yrs?)", s)
    if m:
        val = float(m.group(1))
        return int(val) if val.is_integer() else val

    # Date range: "2589-2566 BC"
    nums = re.findall(r"\d{2,4}", s)

    if len(nums) >= 2:
        a = float(nums[0])
        b = float(nums[1])
        diff = abs(a - b)

        if 1 <= diff <= 100:
            return int(diff) if diff.is_integer() else diff

    return np.nan


def get_pharaoh_column(df: pd.DataFrame) -> str | None:
    return first_existing_column(
        df,
        [
            "pharaoh_final",
            "Pharaoh",
            "pharaoh",
            "Ancient name",
            "ancient_name",
            "Attributed to",
            "attributed_to",
            "owner",
        ],
    )


def get_existing_reign_proxy_columns(df: pd.DataFrame) -> list[str]:
    candidates = [
        "reign_years",
        "pharaoh_reign_years",
        "reign_duration_years",
        "Reign length",
        "reign_length",
        "Reigned",
        "reigned",
        "Reign",
        "reign",
        "reign_dates",
        "Pharaoh reign",
        "pharaoh_reign",
    ]

    return [c for c in candidates if c in df.columns]


def build_reign_proxy_lookup_from_dataset(df: pd.DataFrame) -> dict[str, float]:
    """
    Builds a pharaoh -> reign duration lookup from existing dataset fields,
    if such fields exist.
    """
    pharaoh_col = get_pharaoh_column(df)

    if pharaoh_col is None:
        return {}

    reign_cols = get_existing_reign_proxy_columns(df)

    if not reign_cols:
        return {}

    lookup = {}

    for _, row in df.iterrows():
        pharaoh = normalize_text(row.get(pharaoh_col, ""))

        if not pharaoh:
            continue

        for col in reign_cols:
            proxy = extract_reign_proxy_from_text(row.get(col, np.nan))

            if pd.notna(proxy):
                lookup[pharaoh] = proxy
                break

    return lookup


def manual_pharaoh_reign_proxy_lookup() -> dict[str, float]:
    """
    Manual reign-window proxy lookup.

    These values are NOT direct construction durations.
    They are approximate upper-bound reign-window proxies.
    """
    raw = {
        # 3rd Dynasty
        "Djoser": 19,
        "Sekhemkhet": 6,
        "Khaba": 6,
        "Huni": 24,

        # 4th Dynasty
        "Sneferu": 24,
        "Khufu": 27,
        "Djedefre": 8,
        "Khafre": 26,
        "Khafra": 26,
        "Menkaure": 18,
        "Shepseskaf": 4,

        # 5th Dynasty
        "Userkaf": 7,
        "Sahure": 13,
        "Neferirkare": 11,
        "Neferirkare Kakai": 11,
        "Neferefre": 3,
        "Raneferef": 3,
        "Shepseskare": 1,
        "Nyuserre": 30,
        "Nyuserre Ini": 30,
        "Menkauhor": 8,
        "Menkauhor Kaiu": 8,
        "Djedkare": 38,
        "Djedkare Isesi": 38,
        "Djedkare-Isesi": 38,
        "Unas": 30,

        # 6th Dynasty
        "Teti": 12,
        "Userkare": 2,
        "Pepi I": 49,
        "Merenre": 9,
        "Merenre Nemtyemsaf I": 9,
        "Pepi II": 64,
        "Merenre Nemtyemsaf II": 1,
        "Nitocris": 2,

        # Middle Kingdom
        "Mentuhotep II": 51,
        "Amenemhat I": 30,
        "Senusret I": 45,
        "Sesostris I": 45,
        "Amenemhat II": 35,
        "Senusret II": 19,
        "Sesostris II": 19,
        "Senusret III": 39,
        "Sesostris III": 39,
        "Amenemhat III": 45,
        "Amenemhat IV": 9,
        "Sobekneferu": 4,
        "Khendjer": 5,
        "Ameny Qemau": 2,
        "Mazghuna": 4,

        # New Kingdom / later pyramid-like royal monuments
        "Ahmose": 25,
        "Ahmose I": 25,
    }

    return {normalize_text(k): v for k, v in raw.items()}


def get_pharaoh_value_for_row(row: pd.Series, pharaoh_col: str | None) -> str:
    """
    Gets the attributed pharaoh text for one row.
    """
    if pharaoh_col is not None and pharaoh_col in row.index:
        val = row.get(pharaoh_col, "")

        if pd.notna(val) and str(val).strip():
            return str(val).strip()

    # Fallback: try pyramid name.
    for c in ["name_final", "Modern name", "name"]:
        if c in row.index and pd.notna(row.get(c)):
            return str(row.get(c))

    return ""


def lookup_reign_proxy_for_row(
    row: pd.Series,
    pharaoh_col: str | None,
    dataset_lookup: dict[str, float],
    manual_lookup: dict[str, float],
) -> tuple[float, str]:
    """
    Returns:
        proxy_years, pharaoh_label
    """
    pharaoh_raw = get_pharaoh_value_for_row(row, pharaoh_col)
    pharaoh_norm = normalize_text(pharaoh_raw)

    if not pharaoh_norm:
        return np.nan, ""

    # Exact dataset lookup.
    if pharaoh_norm in dataset_lookup:
        return dataset_lookup[pharaoh_norm], pharaoh_raw

    # Exact manual lookup.
    if pharaoh_norm in manual_lookup:
        return manual_lookup[pharaoh_norm], pharaoh_raw

    # Substring matching.
    for key, val in manual_lookup.items():
        if key and (key in pharaoh_norm or pharaoh_norm in key):
            return val, pharaoh_raw

    return np.nan, pharaoh_raw


def fill_missing_pyramid_durations_from_pharaoh_reign_proxy(df: pd.DataFrame) -> pd.DataFrame:
    """
    Fills remaining missing pyramid durations using reign-window proxy.

    Applies only when:
      - no numeric duration is already available
      - row is not unfinished / abandoned / never-built / incomplete
      - a pharaoh/reign proxy can be found

    It does NOT overwrite direct/manual durations.
    It does NOT set construction_duration_best_years.
    It only fills construction_duration_high_years as an upper-bound proxy.
    """
    df = df.copy()
    df = ensure_duration_columns(df)

    pharaoh_col = get_pharaoh_column(df)
    dataset_lookup = build_reign_proxy_lookup_from_dataset(df)
    manual_lookup = manual_pharaoh_reign_proxy_lookup()

    low = pd.to_numeric(df["construction_duration_low_years"], errors="coerce")
    high = pd.to_numeric(df["construction_duration_high_years"], errors="coerce")
    best = pd.to_numeric(df["construction_duration_best_years"], errors="coerce")

    has_any_numeric_duration = low.notna() | high.notna() | best.notna()

    # Protect unfinished / abandoned / never-built rows.
    text_cols = [
        c for c in [
            "duration_status",
            "duration_method",
            "duration_statement",
            "duration_caution",
            "Comment",
            "comment",
            "status_final",
            "Status",
        ]
        if c in df.columns
    ]

    combined_text = pd.Series("", index=df.index)

    for c in text_cols:
        combined_text = (
            combined_text
            + " "
            + df[c].fillna("").astype(str).map(normalize_text)
        )

    unfinished_mask = combined_text.str.contains(
        r"\b(unfinished|abandoned|never built|never completed|incomplete)\b",
        regex=True,
        na=False,
    )

    fill_mask = (
        ~has_any_numeric_duration
        & ~unfinished_mask
    )

    applied = 0
    no_proxy = []

    for idx in df.index[fill_mask]:
        proxy, pharaoh_label = lookup_reign_proxy_for_row(
            df.loc[idx],
            pharaoh_col=pharaoh_col,
            dataset_lookup=dataset_lookup,
            manual_lookup=manual_lookup,
        )

        if pd.isna(proxy):
            row_name = df.loc[idx, "name_final"] if "name_final" in df.columns else idx
            no_proxy.append(row_name)
            continue

        proxy_out = int(proxy) if float(proxy).is_integer() else float(proxy)

        df.loc[idx, "construction_duration_low_years"] = np.nan
        df.loc[idx, "construction_duration_high_years"] = proxy_out
        df.loc[idx, "construction_duration_best_years"] = np.nan

        df.loc[idx, "duration_status"] = "reign_window_proxy_only"
        df.loc[idx, "duration_method"] = "auto-filled upper-bound proxy from attributed pharaoh reign length"
        df.loc[idx, "duration_confidence"] = "low"
        df.loc[idx, "duration_source_layer"] = "manual_pharaoh_reign_proxy_lookup"

        df.loc[idx, "duration_statement"] = (
            f"No direct construction duration was available. "
            f"The pyramid is attributed to {pharaoh_label}, so this row uses only an upper-bound "
            f"reign-window proxy of about {proxy_out} years."
        )

        df.loc[idx, "source_urls"] = ""

        df.loc[idx, "duration_caution"] = (
            "Auto-filled reign-window proxy. This is not a direct construction duration. "
            "It is an upper-bound proxy based on the attributed pharaoh's reign length. "
            "Use only when a numeric duration is required for comparative analysis."
        )

        applied += 1

    print("\nAuto reign-window proxy fill:")
    print("Pharaoh column:", pharaoh_col)
    print("Dataset-derived pharaoh reign proxies:", len(dataset_lookup))
    print("Manual pharaoh reign proxies:", len(manual_lookup))
    print("Rows eligible for proxy fill:", int(fill_mask.sum()))
    print("Rows filled with reign-window proxy:", applied)
    print("Rows still without proxy:", len(no_proxy))

    if no_proxy:
        print("\nRows still missing proxy because no pharaoh/reign match was found:")
        for x in no_proxy[:30]:
            print(" -", x)

        if len(no_proxy) > 30:
            print(" ...", len(no_proxy) - 30, "more")

    return df


# ============================================================
# MAIN MANUAL CORRECTION FUNCTION
# ============================================================

def manually_edit_pyramid_duration_file(
    input_csv: str | Path,
    output_csv: str | Path | None = None,
) -> pd.DataFrame:
    """
    Manually edits the duration fields in a previously generated pyramid result CSV.

    Direct construction durations are preserved where they were reviewed.
    Remaining missing durations are filled with reign-window proxy only when:
      - no numeric duration exists
      - the row is not unfinished/abandoned
      - the attributed pharaoh can be matched to a reign proxy.
    """
    input_csv = Path(input_csv)

    if output_csv is None:
        output_csv = input_csv
    else:
        output_csv = Path(output_csv)

    df = read_csv_safely(input_csv)
    df = ensure_duration_columns(df)

    # --------------------------------------------------------
    # A) Confirmed / retained direct construction durations
    # --------------------------------------------------------

    df = set_manual_direct_duration(
        df,
        names=[
            "Step Pyramid of Djoser",
            "Pyramid of Djoser",
        ],
        low=19,
        high=19,
        status="manual_source_checked_estimate",
        confidence="medium",
        statement=(
            "Manual curated construction-duration estimate: the Step Pyramid of Djoser "
            "is retained as approximately 19 years."
        ),
    )

    df = set_manual_direct_duration(
        df,
        names=[
            "Red Pyramid",
            "The Red Pyramid",
        ],
        low=10,
        high=17,
        status="manual_source_checked_range",
        confidence="medium-low",
        statement=(
            "Manual curated construction-duration range: the Red Pyramid is retained "
            "as approximately 10–17 years because published estimates differ."
        ),
    )

    df = set_manual_direct_duration(
        df,
        names=[
            "The Great Pyramid of Giza",
            "Great Pyramid of Giza",
            "Pyramid of Khufu",
        ],
        low=20,
        high=27,
        status="manual_source_checked_range",
        confidence="medium",
        statement=(
            "Manual curated construction-duration range: the Great Pyramid of Giza is revised "
            "from a single 26-year estimate to an approximate 20–27 year range."
        ),
    )

    # --------------------------------------------------------
    # B) Downgrade weak direct durations back to proxy-only
    # --------------------------------------------------------

    df = set_manual_reign_proxy(
        df,
        names=[
            "Pyramid of Senusret III",
            "Pyramid of Sesostris III",
        ],
        proxy_high=39,
        pharaoh_label="Senusret III",
        confidence="low",
        caution_extra=(
            "The previously used 20-year value was not kept as a direct construction duration "
            "because the reviewed support was not strong enough."
        ),
    )

    df = set_manual_reign_proxy(
        df,
        names=[
            "Pyramid of Ahmose",
            "Pyramid of Ahmose I",
        ],
        proxy_high=25,
        pharaoh_label="Ahmose I",
        confidence="low",
        caution_extra=(
            "The previously used 10–30 year range was not kept as a direct construction duration "
            "because the reviewed support was not strong enough."
        ),
    )

    # --------------------------------------------------------
    # C) Manual reign-proxy corrections / clarifications
    # --------------------------------------------------------

    df = set_manual_reign_proxy(
        df,
        names=[
            "Pyramid of Khafre",
            "Pyramid of Khafra",
        ],
        proxy_high=26,
        pharaoh_label="Khafre / Khafra",
        confidence="low",
        caution_extra="Alias correction: Khafra is treated as Khafre.",
    )

    df = set_manual_reign_proxy(
        df,
        names=[
            "Pyramid of Neferirkare",
            "Pyramid of Neferirkare Kakai",
        ],
        proxy_high=11,
        pharaoh_label="Neferirkare Kakai",
        confidence="low",
        caution_extra=(
            "Manual correction: older 20-year reign traditions are treated cautiously; "
            "the proxy is revised downward to about 11 years based on the reviewed chronology."
        ),
    )

    df = set_manual_reign_proxy(
        df,
        names=[
            "Pyramid of Djedkare-Isesi",
            "Pyramid of Djedkare Isesi",
        ],
        proxy_high=38,
        pharaoh_label="Djedkare-Isesi",
        confidence="low",
        caution_extra=(
            "This reign length is debated. The value is retained only as a low-confidence "
            "proxy, not a confirmed construction duration."
        ),
    )

    df = set_manual_reign_proxy(
        df,
        names=[
            "Pyramid of Senusret II",
            "Pyramid of Sesostris II",
        ],
        proxy_high=19,
        pharaoh_label="Senusret II",
        confidence="low",
        caution_extra=(
            "Traditional and modern chronologies differ. This value is retained only as a "
            "low-confidence reign-window proxy."
        ),
    )

    df = set_manual_reign_proxy(
        df,
        names=[
            "Pyramid of Pepi I",
        ],
        proxy_high=49,
        pharaoh_label="Pepi I",
        confidence="low",
        caution_extra=(
            "Kept as proxy-only because it was not reconfirmed as a direct construction duration."
        ),
    )

    df = set_manual_reign_proxy(
        df,
        names=[
            "Pyramid of Merenre",
            "Pyramid of Merenre Nemtyemsaf I",
        ],
        proxy_high=9,
        pharaoh_label="Merenre Nemtyemsaf I",
        confidence="low",
        caution_extra=(
            "Kept as proxy-only because it was not reconfirmed as a direct construction duration."
        ),
    )

    # --------------------------------------------------------
    # D) Fill remaining missing durations with reign-window proxy
    # --------------------------------------------------------
    df = fill_missing_pyramid_durations_from_pharaoh_reign_proxy(df)

    df.to_csv(output_csv, index=False)

    print("\nManual duration edits complete.")
    print("Input file: ", input_csv)
    print("Saved file: ", output_csv)
    print("Rows:", len(df), "Columns:", df.shape[1])

    print("\nDuration status counts:")
    print(df["duration_status"].value_counts(dropna=False))

    print("\nNumeric duration coverage:")
    print("low years:", int(pd.to_numeric(df["construction_duration_low_years"], errors="coerce").notna().sum()))
    print("high years:", int(pd.to_numeric(df["construction_duration_high_years"], errors="coerce").notna().sum()))
    print("best years:", int(pd.to_numeric(df["construction_duration_best_years"], errors="coerce").notna().sum()))

    return df


# ============================================================
# RUN
# ============================================================

PROJECT_DIR = Path("/Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data")

input_file = PROJECT_DIR / "pyramids_kaggle_clean.csv"
output_file = PROJECT_DIR / "pyramids_kaggle_clean_manual_duration_corrected.csv"

df_corrected = manually_edit_pyramid_duration_file(
    input_csv=input_file,
    output_csv=output_file,
)


Auto reign-window proxy fill:
Pharaoh column: pharaoh_final
Dataset-derived pharaoh reign proxies: 0
Manual pharaoh reign proxies: 50
Rows eligible for proxy fill: 19
Rows filled with reign-window proxy: 8
Rows still without proxy: 11

Rows still missing proxy because no pharaoh/reign match was found:
 - Pyramid G1-a
 - Pyramid G1-b
 - Pyramid G1-c
 - Step tomb of Khentkaus I
 - Pyramid of Neferhetepes (wife of Userkaf)
 - Pyramid of Queen Khentkawes II
 - Pyramid of Ibi
 - Pyramid of Khui
 - Southern South Saqqara pyramid
 - Tomb S9
 - Tomb S10

Manual duration edits complete.
Input file:  /Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/pyramids_kaggle_clean.csv
Saved file:  /Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/pyramids_kaggle_clean_manual_duration_corrected.csv
Rows: 62 Columns: 41

Duration status counts:
duration_status
reign_window_proxy_only           40
duration_unknown                  11
unfinished_duration_unknown        8
manual_sour

/var/folders/8_/ww6tchkd18sdf8vp6yh6l11w0000gn/T/ipykernel_46322/827215549.py:123: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, "source_urls"] = source_urls
/var/folders/8_/ww6tchkd18sdf8vp6yh6l11w0000gn/T/ipykernel_46322/827215549.py:468: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  unfinished_mask = combined_text.str.contains(
